# 📊 Compare All Models (LITE Version)
**Run this AFTER running all 3 LITE model notebooks**

Upload the result folders from each notebook as Kaggle datasets:
- `SALIENT_Results/`
- `CNN_LSTM_Results/`
- `Transformer_Results/`

**No GPU needed** — this runs in seconds on CPU.

---

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, glob

WINDOW_SIZE = '30sec_150ts'

# Load results — supports both verbose (nested) and legacy (flat) pickle formats
results = {}

# Try multiple path patterns (Kaggle dataset uploads or local working dirs)
search_paths = {
    'SALIENT': [
        '/kaggle/input/salient-results/SALIENT_Results/',
        '/kaggle/working/SALIENT_Results/',
        '/kaggle/input/salient-results/',
    ],
    'CNN-LSTM': [
        '/kaggle/input/cnn-lstm-results/CNN_LSTM_Results/',
        '/kaggle/working/CNN_LSTM_Results/',
        '/kaggle/input/cnn-lstm-results/',
    ],
    'Transformer': [
        '/kaggle/input/transformer-results/Transformer_Results/',
        '/kaggle/working/Transformer_Results/',
        '/kaggle/input/transformer-results/',
    ],
}

def load_result(name, paths):
    for base in paths:
        verbose_path = os.path.join(base, f'results_{WINDOW_SIZE}.pkl')
        if os.path.exists(verbose_path):
            with open(verbose_path, 'rb') as f:
                data = pickle.load(f)
            if 'results' in data and 'overall_accuracy' in data['results']:
                r = data['results']
                return {
                    'accuracy': r['overall_accuracy'],
                    'f1': r['overall_f1'],
                    'kappa': r['overall_kappa'],
                    'cm': r['confusion_matrix'],
                    'subjects': r.get('subject_results', []),
                    'mean_acc': r.get('mean_accuracy'),
                    'std_acc': r.get('std_accuracy'),
                    'ci_95': r.get('ci_95'),
                }
            return data
        
        legacy_names = [f'{name.replace("-","_")}_results.pkl', f'{name}_results.pkl']
        for fname in legacy_names:
            legacy_path = os.path.join(base, fname)
            if os.path.exists(legacy_path):
                with open(legacy_path, 'rb') as f:
                    return pickle.load(f)
        
        if os.path.isdir(base):
            for pkl in glob.glob(os.path.join(base, '*.pkl')):
                with open(pkl, 'rb') as f:
                    return pickle.load(f)
    return None

for name, paths in search_paths.items():
    data = load_result(name, paths)
    if data:
        results[name] = data
    else:
        print(f"\u26a0\ufe0f {name} results not found")

print(f"\u2705 Loaded {len(results)} models: {list(results.keys())}")

In [ ]:
# Comparison Table
print("\n" + "="*60)
print("MODEL COMPARISON (LITE VERSION)")
print("="*60)
print(f"{'Model':<15} {'Accuracy':<12} {'F1':<12} {'Kappa':<10}")
print("-"*50)

for name, res in results.items():
    acc = res['accuracy'] * 100
    f1 = res['f1'] * 100
    kappa = res['kappa']
    extra = ""
    if res.get('mean_acc') and res.get('std_acc'):
        extra = f"  ({res['mean_acc']*100:.1f}% \u00b1 {res['std_acc']*100:.1f}%)"
    print(f"{name:<15} {acc:.2f}%{'':<5} {f1:.2f}%{'':<5} {kappa:.4f}{extra}")

print("-"*50)
print(f"{'Tufts RF':<15} 72.20%       (benchmark)")
print("="*60)

best = max(results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n\ud83c\udfc6 Best Model: {best[0]} ({best[1]['accuracy']*100:.2f}%)")

if best[1].get('ci_95'):
    mean = best[1]['mean_acc'] * 100
    ci = best[1]['ci_95'] * 100
    print(f"   95% CI: [{mean-ci:.2f}%, {mean+ci:.2f}%]")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = {'SALIENT': '#2E86AB', 'CNN-LSTM': '#F18F01', 'Transformer': '#2EAB5E'}
models = list(results.keys())

# 1. Accuracy Bar Chart
ax = axes[0]
accs = [results[m]['accuracy']*100 for m in models]
bars = ax.bar(models, accs, color=[colors.get(m, '#888') for m in models])
ax.axhline(72.2, color='red', linestyle='--', linewidth=2, label='Tufts Benchmark (72.2%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Comparison')
ax.set_ylim(0, 100)
ax.legend()
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{acc:.1f}%', ha='center', fontweight='bold')

# 2. F1 Score
ax = axes[1]
f1s = [results[m]['f1']*100 for m in models]
bars = ax.bar(models, f1s, color=[colors.get(m, '#888') for m in models])
ax.set_ylabel('F1 Score (%)')
ax.set_title('F1 Score Comparison')
ax.set_ylim(0, 100)
for bar, f1 in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{f1:.1f}%', ha='center', fontweight='bold')

# 3. Confusion Matrix for best model
ax = axes[2]
best_name = best[0]
cm = np.array(results[best_name]['cm'])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues', ax=ax,
            xticklabels=['0-back','1-back','2-back','3-back'],
            yticklabels=['0-back','1-back','2-back','3-back'])
ax.set_title(f'{best_name} Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('/kaggle/working/model_comparison.png', dpi=300)
plt.show()
print("\u2705 Saved: model_comparison.png")

In [ ]:
# Per-Subject Accuracy Comparison
has_subjects = all(res.get('subjects') for res in results.values())

if has_subjects:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(1, len(list(results.values())[0]['subjects']) + 1)
    width = 0.25
    
    for i, (name, res) in enumerate(results.items()):
        subj_accs = [s['accuracy']*100 for s in res['subjects']]
        ax.bar(x + i*width, subj_accs, width, label=name, color=colors.get(name, '#888'), alpha=0.8)
    
    ax.axhline(72.2, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Tufts RF (72.2%)')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Per-Subject Accuracy Comparison')
    ax.set_xticks(x + width)
    ax.set_xticklabels([f'S{i}' for i in x])
    ax.legend()
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.savefig('/kaggle/working/per_subject_comparison.png', dpi=300)
    plt.show()
    print("\u2705 Saved: per_subject_comparison.png")

# LaTeX Table for Paper
print("\n\ud83d\udcc4 LaTeX Table for Paper:\n")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Model Comparison on Tufts fNIRS2MW Dataset (LOSO CV)}")
print(r"\begin{tabular}{lccc}")
print(r"\hline")
print(r"Model & Accuracy (\%) & F1 Score (\%) & Cohen's $\kappa$ \\")
print(r"\hline")
for name, res in results.items():
    print(f"{name} & {res['accuracy']*100:.2f} & {res['f1']*100:.2f} & {res['kappa']:.4f} \\\\")
print(r"\hline")
print(r"Tufts RF (Benchmark) & 72.20 & - & - \\")
print(r"\hline")
print(r"\end{tabular}")
print(r"\end{table}")